# Suite1 Project4 Tuning

**Dataset:** California Housing

---
Run each cell in order. All outputs are generated from real data.

In [ ]:
# Setup: imports and configuration
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Plotting style
sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams['figure.dpi'] = 120

# Helper: load dataset
def load_data(name):
    return pd.read_csv(f'https://raw.githubusercontent.com/ulink-ibcs/A4_ML_Projects/main/datasets/{name}')

print("✅ Setup complete!")

In [ ]:
"""Project 4: Hyperparameter Tuning & Validation"""
import sys, os; sys.path.insert(0, os.path.dirname(__file__))

df = load_csv('housing.csv')
OUTPUT_DIR = os.path.join(BASE_DIR, 'suite1_project4_outputs'); os.makedirs(OUTPUT_DIR, exist_ok=True)

## PROJECT 4: HYPERPARAMETER TUNING & MODEL VALIDATION

In [ ]:
X = df[['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']]
y = df['MedHouseVal']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ── 1. Train/Test vs Train/Validation/Test ──

## 1. The Problem: Splitting Data

In [ ]:
X_train_v, X_val, y_train_v, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=42)
print(f"Final split: Train={X_train_v.shape[0]:,} | Validation={X_val.shape[0]:,} | Test={X_test.shape[0]:,}")
print(f"Ratio: {X_train_v.shape[0]/len(df):.0%} | {X_val.shape[0]/len(df):.0%} | {X_test.shape[0]/len(df):.0%}")

# Demonstrate overfitting
print("\n=== Overfitting Demonstration ===")
model_deep = Pipeline([
    ('scaler', StandardScaler()),
    ('ridge', Ridge(alpha=0.0001))
])
model_deep.fit(X_train, y_train)
train_r2 = r2_score(y_train, model_deep.predict(X_train))
test_r2 = r2_score(y_test, model_deep.predict(X_test))
print(f"Ridge(alpha=0.0001): Train R²={train_r2:.4f}, Test R²={test_r2:.4f}")
print(f"  Gap = {train_r2 - test_r2:.4f} → overfitting!")

model_regularized = Ridge(alpha=10)
model_regularized.fit(X_train, y_train)
train_r2_r = r2_score(y_train, model_regularized.predict(X_train))
test_r2_r = r2_score(y_test, model_regularized.predict(X_test))
print(f"\nRidge(alpha=10):    Train R²={train_r2_r:.4f}, Test R²={test_r2_r:.4f}")
print(f"  Gap = {train_r2_r - test_r2_r:.4f} → better generalization")

# ── 2. Cross-Validation ──

## 2. K-Fold Cross-Validation

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

model = Ridge(alpha=1.0)
cv_scores = cross_val_score(model, X_scaled, y, cv=5, scoring='r2')
print(f"5-Fold CV R² scores: {[f'{s:.4f}' for s in cv_scores]}")
print(f"Mean CV R²: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print(f"Range: {cv_scores.min():.4f} – {cv_scores.max():.4f}")

# Visualize CV scores
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(range(1, 6), cv_scores, color='#2e86de', alpha=0.7, edgecolor='white')
ax.axhline(y=cv_scores.mean(), color='#e74c3c', linestyle='--', linewidth=2, label=f'Mean = {cv_scores.mean():.4f}')
ax.fill_between(range(1, 6), cv_scores.mean()-cv_scores.std(), cv_scores.mean()+cv_scores.std(), 
                alpha=0.15, color='#e74c3c', label=f'±1 Std = {cv_scores.std():.4f}')
ax.set_xlabel('Fold', fontsize=12); ax.set_ylabel('R² Score', fontsize=12)
ax.set_title('5-Fold Cross-Validation Results', fontsize=13)
ax.set_xticks(range(1, 6)); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
# 'p4_cv_scores')
plt.show()
print("[Saved] p4_cv_scores.png")

# ── 3. Grid Search ──

## 3. Grid Search — Finding the Best Alpha

In [ ]:
param_grid = {'alpha': [0.001, 0.01, 0.1, 0.5, 1, 5, 10, 50, 100, 500]}
grid = GridSearchCV(Ridge(), param_grid, cv=5, scoring='r2', return_train_score=True)
grid.fit(X_scaled, y)

print(f"Best alpha: {grid.best_params_['alpha']}")
print(f"Best CV R²: {grid.best_score_:.4f}")

cv_results = pd.DataFrame(grid.cv_results_)
print(f"\nTop 5 configurations:")
cols = ['param_alpha', 'mean_test_score', 'mean_train_score', 'std_test_score']
print(cv_results.sort_values('mean_test_score', ascending=False)[cols].head().to_string(index=False))

# Learning curves
fig, ax = plt.subplots(figsize=(10, 6))
train_sizes, train_scores, test_scores = learning_curve(
    Ridge(alpha=grid.best_params_['alpha']), X_scaled, y, 
    train_sizes=np.linspace(0.1, 1.0, 10), cv=5, scoring='r2')
train_mean = np.mean(train_scores, axis=1); train_std = np.std(train_scores, axis=1)
test_mean = np.mean(test_scores, axis=1); test_std = np.std(test_scores, axis=1)

ax.plot(train_sizes, train_mean, 'o-', color='#2e86de', label='Training Score')
ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.15, color='#2e86de')
ax.plot(train_sizes, test_mean, 'o-', color='#e74c3c', label='CV Score')
ax.fill_between(train_sizes, test_mean - test_std, test_mean + test_std, alpha=0.15, color='#e74c3c')
ax.set_xlabel('Training Examples', fontsize=12)
ax.set_ylabel('R² Score', fontsize=12)
ax.set_title(f'Learning Curve — Ridge(alpha={grid.best_params_["alpha"]})', fontsize=13)
ax.legend(loc='lower right'); ax.grid(True, alpha=0.3)
plt.tight_layout()
# 'p4_learning_curve')
plt.show()
print("[Saved] p4_learning_curve.png")

# ── 4. Grid Search Visualization ──

## 4. Grid Search Visualization

In [ ]:
results_df = cv_results[['param_alpha', 'mean_test_score', 'mean_train_score', 'std_test_score']]
results_df.columns = ['Alpha', 'Test R²', 'Train R²', 'Std']

fig, ax = plt.subplots(figsize=(10, 6))
ax.semilogx(results_df['Alpha'], results_df['Test R²'], 'o-', color='#2e86de', linewidth=2, label='Test (CV)')
ax.semilogx(results_df['Alpha'], results_df['Train R²'], 's--', color='#e74c3c', linewidth=2, label='Train')
ax.axvline(x=grid.best_params_['alpha'], color='green', linestyle=':', alpha=0.7, label=f'Best α={grid.best_params_["alpha"]}')
ax.fill_between(results_df['Alpha'], 
                results_df['Test R²'] - results_df['Std'], 
                results_df['Test R²'] + results_df['Std'], 
                alpha=0.15, color='#2e86de')
ax.set_xlabel('Alpha (regularization strength)', fontsize=12)
ax.set_ylabel('R² Score', fontsize=12)
ax.set_title('Grid Search: Effect of Alpha on Model Performance', fontsize=13)
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
# 'p4_grid_search')
plt.show()
print("[Saved] p4_grid_search.png")

# ── 5. Final Evaluation ──

## 5. Final Model Evaluation on Test Set

In [ ]:
best_model = grid.best_estimator_
y_pred = best_model.predict(scaler.transform(X_test))
final_r2 = r2_score(y_test, y_pred)
final_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"Final model (Ridge, α={grid.best_params_['alpha']}) on test set:")
print(f"  R²:   {final_r2:.4f}")
print(f"  RMSE: {final_rmse:.4f} ($100K)")

results_final = {
    'best_alpha': float(grid.best_params_['alpha']),
    'best_cv_score': float(grid.best_score_),
    'final_test_r2': float(final_r2),
    'final_test_rmse': float(final_rmse),
    'cv_5fold_scores': [float(s) for s in cv_scores],
    'cv_mean': float(cv_scores.mean()),
    'cv_std': float(cv_scores.std()),
    'grid_search_results': cv_results[['param_alpha', 'mean_test_score', 'mean_train_score']].to_dict('records'),
}
    json.dump(results_final, f, indent=2, default=str)

print("Done.")